## Disambiguating Census Division Names

Our original census division (CD) commuting data contains ambiguous census division names and it's causing problems.

Let's make a dataframe containing the parent province for every CD by DGUID, and then use this to fully disambiguate the names in the original commuting CSV, appending the province names to the census division names.

Example: In our commuting CSV, in rows where the `DGUID` value is `2021A00031001`, the `GEO` value is currently `Division No. 1`.

In this case, we can Google search and find out that the CD with DGUID `2021A00031001` is in Newfoundland. But this data isn't in our original commuting CSV file.

So, in our commuting CSV, we will rename all the rows with DGUID `2021A00031001` from

`Division No. 1`

to

`Division No. 1, Newfoundland and Labrador`

For this, we've downloaded a metadata file from StatsCan, the [2021 Census Geographic Attribute File
](https://www12.statcan.gc.ca/census-recensement/2021/geo/aip-pia/attribute-attribs/index2021-eng.cfm), which contains the data we need.

Derek has placed this metadata file at `REPO_ROOT/../original_data/metadata/census_divisions_metadata.csv` on his laptop.

Why? The file is 200+ MB and thus too big to commit to the repo. If you want to run this notebook, do this:
1. Download the CSV from the link above.
2. In the folder that contains this repo, make this directory structure: `original_data/metadata/`. So, this will exist alongside the repo folder.
3. Name the CSV `census_divisions_metadata.csv` and put it at `original_data/metadata/census_divisions_metadata.csv`

Note that in the cell below (which creates the subsetted metadata), the relative path to the original (200+ MB) metadata CSV assumes you have done this!

In [1]:
import pandas as pd
import os

# Load the Metadata CSV from statscan
script_dir = os.getcwd()
metadata_path = os.path.abspath(os.path.join(script_dir, "../../original_data/metadata/census_divisions_metadata.csv"))
df = pd.read_csv(metadata_path, dtype=str, encoding="latin1") # there are french characters, so we need to use latin1 encoding

# Select only relevant columns
df_filtered = df[["PRENAME_PRANOM", "CDUID_DRIDU", "CDDGUID_DRIDUGD", "CDNAME_DRNOM", "CSDNAME_SDRNOM"]]

# Drop duplicate CDUID_DRIDU values, keeping only the first occurrence
df_unique = df_filtered.drop_duplicates(subset="CDUID_DRIDU", keep="first")

# Save to CSV (optional)
output_path = os.path.abspath(os.path.join(script_dir, "../data/processed/metadata/census_divisions_metadata_filtered_unique_cduids.csv"))
df_unique.to_csv(output_path, index=False, encoding="latin1")

df_unique

,PRENAME_PRANOM,CDUID_DRIDU,CDDGUID_DRIDUGD,CDNAME_DRNOM,CSDNAME_SDRNOM
0,Newfoundland and Labrador,1001,2021A00031001,Division No. 1,St. John's
3664,Newfoundland and Labrador,1002,2021A00031002,Division No. 2,"Division No. 2, Subd. K"
4118,Newfoundland and Labrador,1003,2021A00031003,Division No. 3,"Division No. 3, Subd. A"
4393,Newfoundland and Labrador,1004,2021A00031004,Division No. 4,Gallants
4860,Newfoundland and Labrador,1005,2021A00031005,Division No. 5,Jackson's Arm
...,...,...,...,...,...
497377,Northwest Territories,6105,2021A00036105,Region 5,Fort Resolution
497713,Northwest Territories,6106,2021A00036106,Region 6,Dettah
498008,Nunavut,6204,2021A00036204,Qikiqtaaluk,Grise Fiord
498368,Nunavut,6205,2021A00036205,Kivalliq,Naujaat


Note, for the cell below, we need the `chardet` package installed. I've added it to `environment.yml`.

In [2]:
import pandas as pd
import os

# Define file paths
script_dir = os.getcwd()
commuting_data_path = os.path.abspath(os.path.join(script_dir, "../data/raw/commuting_data/commuting_data_census_divisions.csv"))
output_path = os.path.abspath(os.path.join(script_dir, "../data/processed/commuting_data/commuting_data_census_divisions_disambiguated.csv"))

# Step 1: Try loading the file with UTF-8 (since us-ascii is a subset of UTF-8)
try:
    df_commuting = pd.read_csv(commuting_data_path, dtype=str, encoding="utf-8")
    detected_encoding = "utf-8"
    print("Loaded file successfully with UTF-8 encoding.")
except UnicodeDecodeError:
    print("UTF-8 failed, retrying with Latin-1 encoding.")
    df_commuting = pd.read_csv(commuting_data_path, dtype=str, encoding="latin1")
    detected_encoding = "latin1"

# Create a copy of the original data for later data integrity verification
df_commuting_original = df_commuting.copy()

# Step 2: Create a lookup dictionary from df_unique (DGUID -> Province)
province_lookup = df_unique.set_index("CDDGUID_DRIDUGD")["PRENAME_PRANOM"].to_dict()

# Step 3: Modify the GEO column using the lookup
df_commuting["GEO"] = df_commuting.apply(
    lambda row: f"{row['GEO']}, {province_lookup[row['DGUID']]}" 
    if row["DGUID"] in province_lookup else row["GEO"], axis=1
)

# Step 4: Save the modified file using the detected encoding
df_commuting.to_csv(output_path, index=False, encoding=detected_encoding)

df_commuting

Loaded file successfully with UTF-8 encoding.


,REF_DATE,GEO,DGUID,Time arriving at work (16),Main mode of commuting (21),Coordinate,Commuting duration (7):Total - Commuting duration[1],Symbol,Commuting duration (7):Less than 15 minutes[2],Symbol.1,Commuting duration (7):15 to 29 minutes[3],Symbol.2,Commuting duration (7):30 to 44 minutes[4],Symbol.3,Commuting duration (7):45 to 59 minutes[5],Symbol.4,Commuting duration (7):60 minutes and over[6],Symbol.5,Commuting duration (7):Average commuting duration (in minutes)[7],Symbol.6
0,2021,"Division No. 1, Newfoundland and Labrador",2021A00031001,Total - Time arriving at work,Total - Main mode of commuting,3.1.1,83910.0,NaN,37815.0,NaN,34420.0,NaN,6740.0,NaN,1830.0,NaN,3100.0,NaN,17.3,NaN
1,2021,"Division No. 1, Newfoundland and Labrador",2021A00031001,Total - Time arriving at work,"Car, truck or van",3.1.2,75940.0,NaN,34000.0,NaN,32295.0,NaN,5640.0,NaN,1515.0,NaN,2490.0,NaN,17.1,NaN
2,2021,"Division No. 1, Newfoundland and Labrador",2021A00031001,Total - Time arriving at work,"Car, truck or van - as a driver",3.1.3,68570.0,NaN,29690.0,NaN,29815.0,NaN,5360.0,NaN,1440.0,NaN,2270.0,NaN,17.3,NaN
3,2021,"Division No. 1, Newfoundland and Labrador",2021A00031001,Total - Time arriving at work,Driver (only worker in vehicle),3.1.4,64680.0,NaN,28445.0,NaN,28305.0,NaN,4875.0,NaN,1210.0,NaN,1840.0,NaN,16.9,NaN
4,2021,"Division No. 1, Newfoundland and Labrador",2021A00031001,Total - Time arriving at work,Driver with 1 passenger,3.1.5,3325.0,NaN,1125.0,NaN,1345.0,NaN,405.0,NaN,170.0,NaN,280.0,NaN,22.6,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98443,2021,"Kitikmeot, Nunavut",2021A00036208,Between 12 a.m. and 4:59 a.m.,Active transportation,5460.16.17,20.0,NaN,15.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,11.0,NaN
98444,2021,"Kitikmeot, Nunavut",2021A00036208,Between 12 a.m. and 4:59 a.m.,Walked,5460.16.18,20.0,NaN,15.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,11.0,NaN
98445,2021,"Kitikmeot, Nunavut",2021A00036208,Between 12 a.m. and 4:59 a.m.,Bicycle,5460.16.19,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...
98446,2021,"Kitikmeot, Nunavut",2021A00036208,Between 12 a.m. and 4:59 a.m.,"Motorcycle, scooter or moped",5460.16.20,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...


Let's do a data integrity check to verify that we only modified the `GEO` column:

In [3]:
# Step 1: Drop the GEO column from both DataFrames
df_original_no_geo = df_commuting_original.drop(columns=["GEO"])
df_modified_no_geo = df_commuting.drop(columns=["GEO"])

# Step 2: Compare the two DataFrames
comparison_result = df_original_no_geo.equals(df_modified_no_geo)

# Step 3: Display result
if comparison_result:
    print("Verification successful: All columns except 'GEO' are identical.")
else:
    print("Verification failed: Some columns besides 'GEO' have changed.")

# Step 4 (Optional): Show the differences if any exist
if not comparison_result:
    diff = df_original_no_geo.compare(df_modified_no_geo)
    print("Differences detected:")
    display(diff)

Verification successful: All columns except 'GEO' are identical.
